# Explore Patient Cases
This notebook allows for visualizing real patient cases.

In [ ]:
%%script bash
python3 -m pip list

In [ ]:
%%script bash
. /opt/pyenv/bin/activate && \
python3 -m pip list

# Convert VCF to CSV format

In [ ]:
from glob import glob

In [ ]:
# List of annotated, ranked pathogennic variants in VCF format
pathogenic_variant_vcfs = glob('/rdds/tmp/variant-rank-score/test_cases_explanations/*/pathogenic_variants.vcf')
pathogenic_variant_vcfs

In [ ]:
# Convert VCF files to CSV files
for vcf_path in pathogenic_variant_vcfs:
    !bash -c "echo -e Converting {vcf_path}"
    !bash -c ". /opt/pyenv/bin/activate && PYTHONPATH=/rdds/src python3 -m rdds.lib.vcf2csv convert {vcf_path}"

# Load and Preprocess Pathogenic Variants

In [ ]:
#from pathlib import PurePath
csv_files = glob('/rdds/tmp/variant-rank-score/test_cases_explanations/*/pathogenic_variants.csv')
#csv_files = [PurePath(path) for path in csv_files]
csv_files

In [ ]:
import pandas as pd
pd.options.display.max_columns = None

In [ ]:
from os.path import dirname
cases = dict()
for csv_file in csv_files:
    case_name = dirname(csv_file).split('/')[-1]
    df = pd.read_csv(csv_file, index_col=0, low_memory=False)
    df['case_name'] = case_name
    cases[case_name] = df

In [ ]:
df = None
for _, case_df in cases.items():
    if df is None:
        df = case_df.copy()
    else:
        df = pd.concat((df, case_df), axis=0)
df.reset_index(inplace=True, drop=True)

In [ ]:
df

In [ ]:
annotations = list(df.columns)
annotations.remove('CHROM')
annotations.remove('POS')
annotations.remove('ID')
annotations.remove('ALT')
annotations.remove('REF')
annotations.remove('CSQ_STRAND')
annotations = sorted(annotations)
annotations

# Feature - Rank Score Analysis

In [ ]:
# HACK: Install of seaborn==0.12.2 manually in container
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
FIGSIZE=(10, 5)

In [ ]:
def visualize_numerical_feature_vs_rank_score(df, annotation):
    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_subplot()
    sb.scatterplot(df[[annotation, 'MivmirScore']],
                   x=annotation,
                   y='MivmirScore',
                   ax=ax)
    fig.tight_layout()
    fig.suptitle(f'n={len(df[annotation].dropna())} ({len(df)})')
    plt.show(fig)

In [ ]:
from tensorflow_text import RegexSplitter
import tensorflow as tf

split_regex = '\s|\n|_|&|/|\||:|,|-|0|1|2|3|4|5|6|7|8|9'
preprocessor = RegexSplitter(split_regex=split_regex)
def _preprocess_tensorflow(s: str) -> tf.RaggedTensor:
    assert isinstance(s, str), type(s)
    s = s.lower()
    s = tf.ragged.constant(s)
    s = preprocessor.split(s)
    # Drop bases from text data
    regexp_matches = tf.strings.regex_full_match(input=s, pattern='^[CAGTNcagtn]+$')
    # Invert the matches to preserve non-matching strings
    mask = tf.math.logical_not(regexp_matches)
    # Mask the original tensor to drop strings matching regexp pattern
    s = tf.ragged.boolean_mask(data=s, mask=mask)
    return s

def _preprocess_text(string, as_string=True) -> tf.Tensor:
    if isinstance(string, (float, int)):
        if not (string == string):  # Is NaN value
            return '[N/A]'
        else:
            return 'numtype'
    elif isinstance(string, str):
        if string == np.nan or string is None or len(string) == 0 or string == ' ':
            return '[N/A]'
    else:
        raise ValueError(f'Expected str or num data, got {string}, {type(string)}')
    tensor = _preprocess_tensorflow(string)
    if not as_string:
        return tensor
    tokens: np.ndarray = tensor.numpy()[0]
    # Create a cleaned sentence from tokens
    sentence = ''
    for t in tokens:
        sentence += ' ' + t.decode('utf-8')
    return sentence

def visualize_textual_features_vs_rank_score(df, annotation):
    preprocessed_df = pd.DataFrame()
    data = df[[annotation]]
    for index, row in data.iterrows():
        sentence = _preprocess_text(row[annotation])
        _preprocessed_df = pd.DataFrame(data={'MivmirScore': df.loc[index].MivmirScore,
                                               annotation: sentence,
                                              'case_name': df.loc[index].case_name},
                                        index=[index])
        if preprocessed_df is None:
            preprocessed_df = _preprocessed_df
        else:                       
           preprocessed_df = pd.concat((preprocessed_df, _preprocessed_df), axis=0)
    facet = sb.relplot(data=preprocessed_df, x='case_name', y='MivmirScore', hue=annotation, style=annotation, kind='scatter', height=FIGSIZE[1],
                      aspect = FIGSIZE[0] / FIGSIZE[1])
    facet.set(xticklabels=[])
    facet.set(title=f'n={len(preprocessed_df.dropna())} ({len(df)})')
    facet.tight_layout()

In [ ]:
def print_nan_entries(df, annotation):
    nan_entries = df.loc[df[annotation].isna()][['MivmirScore', annotation]]
    if len(nan_entries) > 0:
        print(nan_entries, flush=True)
        mean = nan_entries.MivmirScore.mean()
        std = nan_entries.MivmirScore.std()
        print(f'mean {mean:.4f} std {std:.4f}')
    return nan_entries.MivmirScore

In [ ]:
IGNORE_ANNOTATIONS = [
    'CSQ_DOMAINS',
    'Compounds_value',
    'Compounds_family_id',
    'CompoundsNormalized',
    'CSQ_ENSP',
    'MivmirExplanation',
    'MivmirScore']

annotation_nan_statistics = {}  # Keep track of pathogenicity scores for empty annotations

for annotation in annotations:
    if annotation in IGNORE_ANNOTATIONS:
        continue
    dtype = df[annotation].dtype
    try:
        if dtype == float or dtype == np.int64:
            visualize_numerical_feature_vs_rank_score(df, annotation=annotation)
        elif dtype == object:
            visualize_textual_features_vs_rank_score(df, annotation)
        else:
            raise ValueError(dtype)
        vrs_model_prediction_for_nan_entries = print_nan_entries(df, annotation)
        annotation_nan_statistics.update({annotation: vrs_model_prediction_for_nan_entries})
    except Exception as e:
        print(f'Error plotting {annotation}')
        raise e

In [ ]:
# Visualize model inference bias in cases where annotations are NaN
df_annotation_nan_statistics = pd.DataFrame(data=annotation_nan_statistics)
for annotation in df_annotation_nan_statistics.keys():
    nan_scores = df_annotation_nan_statistics[annotation].dropna(axis=0)
    if len(nan_scores) == 0:
        continue
    # Plot annotations where the annotation is biased towards benign or pathogenic class (0.5 is neutral)
    d = abs(0.5 - nan_scores.mean())
    if d < 0.2 :  # if bias are less than
        continue
    facet = sb.displot(data=nan_scores, kind='hist', binwidth=0.1)
    facet.set_xlabels('MivmirScore Score')
    facet.set(xlim=(-0.1, 1.1))
    facet.set(title=f'MivmirScore\non {annotation} NaNs n={len(nan_scores)}')
    facet.tight_layout()

In [ ]:
def compute_annotation_magnitude(df, annotation):
    """
    Compute amount of annotation information in this annotation type, for every variant.
    """
    variants = df[[annotation]]
    dtype = variants[annotation].dtype
    max_annotation_length = 0
    max_tokens = None
    annotation_lengths = []
    for index, row in variants.iterrows():
        data = row[annotation]
        length = None
        try:
            data_num = float(data)
            if data_num == data_num: # NaN test
                length = 1
            else:
                length = 0
        except Exception as e:
            sentence = _preprocess_text(data)
            length = len(sentence)
            # TODO: EVALUATE WHETHER SENTENCE LENGTH IS SPECIFIC FOR TARGET METRIC
            length = 1 if length > 0 else 0
        if length > max_annotation_length:
            max_annotation_length = length
            max_tokens = data
        annotation_lengths.append(length)
    if max_annotation_length > 0:
        annotation_lengths = [l / float(max_annotation_length)  for l in annotation_lengths]
    return variants.index, annotation_lengths

df_annotation_info = None
for annotation in annotations:
    index, annotation_lengths = compute_annotation_magnitude(df, annotation)
    _df_annotation_info = pd.DataFrame(data={f'{annotation}%': annotation_lengths}, index=index)
    if df_annotation_info is None:
        df_annotation_info = _df_annotation_info
    else:
        df_annotation_info = pd.concat((df_annotation_info, _df_annotation_info), axis=1)
df_annotation_info['MivmirScore'] = df.MivmirScore
df_annotation_info['case_name'] = df.case_name

for annotation in annotations:
    facet = sb.relplot(data=df_annotation_info, x=f'{annotation}%', y='MivmirScore', kind='scatter', height=FIGSIZE[1],
                      aspect = FIGSIZE[0] / FIGSIZE[1])
    facet.set(title=f'Annotation magnitude {annotation}')
    facet.tight_layout()

In [ ]:
annotation_magnitudes = df_annotation_info.iloc[:, :-5].sum(axis=1)
facet = sb.relplot(x=annotation_magnitudes, y=df_annotation_info.MivmirScore, hue=df_annotation_info.case_name, style=df_annotation_info.case_name, kind='scatter', height=FIGSIZE[1],
                      aspect = FIGSIZE[0] / FIGSIZE[1])
for index, row in df_annotation_info.iterrows():
    facet.ax.text(annotation_magnitudes.loc[index], row.MivmirScore, row.case_name, fontsize=6, rotation=40)
facet.set(xlabel='Total Annotation Magnitude')
facet.set(title=f'Overall annotation magnitude')
facet.tight_layout()
pd.concat((annotation_magnitudes, df_annotation_info[['case_name', 'MivmirScore']]), axis=1).sort_values(0)

In [ ]:
# Investigate missense vs frameshift variant bias
df_investigate_framshift_missense = pd.DataFrame()
df_investigate_framshift_missense['consequence'] = df.CSQ_Consequence
df_investigate_framshift_missense['annotation_magnitude'] = annotation_magnitudes
df_investigate_framshift_missense['MivmirScore'] = df.MivmirScore
df_investigate_framshift_missense['case_name'] = df.case_name
df_investigate_framshift_missense.sort_values('consequence', inplace=True)

facet = sb.relplot(y=df_investigate_framshift_missense.consequence,
                   x=df_investigate_framshift_missense.annotation_magnitude,
                   hue=df_investigate_framshift_missense.MivmirScore,
                   height=FIGSIZE[1],
                   aspect=FIGSIZE[0] / FIGSIZE[1])
for index, row in df_investigate_framshift_missense.iterrows():
    facet.ax.text(row.annotation_magnitude, row.consequence, row.case_name, fontsize=6, rotation=40)
facet.tight_layout()

# Takeaways Above Plots
- Clinvar annotation: Majority of the cases with low score had no clinvar annotation associated.
- Frameshift variants seems to be poorly classified, see CSQ_Consequence
- Missense variants seems to be accurately classified, see CSQ_Consequence
- "/.../ Non coding transcript variant"s seem to be poorly classified, see CSQ_Consequence
- DELETIONS seems to be the most difficult variants to properly classify, see CSQ_HGVSc
- LoFTool seems not to be correlated with pathogenicity score. However, all NaN entries in this annotation is classified as highly pathogenic.
- Most of low scoring variants have N/A in CSQ_Polyphen
- CSQ_REVEL_score and pathogenicity score is almost linear in relationship
- Most of low scoring variants have N/A in CSQ_SIFT
- Most of variants lie in highest range of CSQ_phastConst100way_vertebrate
- Frq, GNOMADAF: inverse relationship than expected; a low freq mostly ends up with low pathogenicity score. Variants without annotations
    in these fields shows no bias to benign/ pathogenic classification.
- Missing SpliceAI annotations; these variants are inccurately scored very benign (n=3)
- CSQ_UNIPARC missing annotations; variants are inaccurately scored very benign.

## Summary
- Missing clinvar annotations have a negative impact on performance
- SIFT, CSQ_Consequence, PolyPhen N/A behavior on non-coding variants (or uncharted proteins) seem to suggest that novel variants
  not previously seen or charted by these tools have a negative impact on performance when these annotations are missing.
- Popfreq annnotations does not seem to have a noticeable impact on performance

## Discussion
- Can it be that the performance advantage in missense variants can be due to SNPs gets additional annotations, compared to the more-difficult to
  characterize frameshift variants (less annotations equals to more benign prediction)? The annotation_magnitude vs variant type seems to suggest this.
## Actions
Suggest to add synthetic data in training step, to make known pathogenic variants look like novel, uncharted variants.

# Network Analysis

In [ ]:
%%time
from pprint import pprint
annotation_meta = {}

numerical_link_sources = []
numerical_link_targets = []
numerical_link_strengths = []

numerical_annotations = annotations
numerical_annotations = ['CADD', 'CSQ_REVEL_score', 'CSQ_phastCons100way_vertebrate']
numerical_annotations = ['CSQ_MaxEntScan_alt',
                  'CSQ_MaxEntScan_diff',
                  'CSQ_MES-SWA_acceptor_alt',
                  'CSQ_MES-SWA_donor_alt',
                  'CSQ_MES-SWA_donor_diff',
                  'CSQ_SpliceAI_pred_DS_AL',
                  'CSQ_SpliceAI_pred_DS_DG',
                  'CSQ_SpliceAI_pred_DS_DL',
                  'CSQ_REVEL_score',
                  'CSQ_LoFtool',
                  'CSQ_GERP++_RS',
                  'CSQ_phastCons100way_vertebrate',
                  'CSQ_phyloP100way_vertebrate',
                  'CADD',
                  #'ModelScore_value',
                  #'SWEGENAF', Missing in data??
                  #'GNOMADAF_popmax', Missing in data
                  'SPIDEX',
                  'CSQ_SpliceAI_pred_DS_AG',
                  #'Frq' Missing in daat
                  ]
#numerical_annotations = ['SPIDEX']

for annotation in numerical_annotations:
    if df[annotation].dtype != float:
        df[annotation].dtype
        continue
    maximum = float(df[annotation].max())
    minimum = float(df[annotation].min())
    if not ((maximum == maximum) and (minimum == minimum)):  # NaN check
        # Skip annotations that contain no data
        continue
    annotation_meta.update({
        annotation: (minimum, maximum)
    })
    # if maximum == minimum then all points are related.
    # Such relationships adds no extra knowledge to the network, so it's ignored.
    if maximum == minimum:
        continue
    # For variants, compute a normalized similarity metric
    threshold = df[annotation].std() / (maximum - minimum)
    #threshold = threshold * 0.25
    threshold = threshold * 0.05
    assert threshold == threshold
    for index, row in df.iterrows():
        for inner_index, inner_row in df.iterrows():
            if index == inner_index:
                continue
            # TODO: Limit what links to create, a capping method. Consider only "related" samples
            src = row[annotation]
            target = inner_row[annotation]
            # metric behavior: 0: identical match, 1.0: large difference
            if not (src == src) or not (target == target):
                # Target or src is NaN
                delta = 1.0
            else:
                delta = abs(src - target)
                delta = (delta) / (maximum - minimum)
            assert delta == delta
            assert 0.0 <= delta <= 1.0, (delta, src, target)
            if delta <= threshold:
                link_strength = 1.0 - delta
                numerical_link_sources.append(index)
                numerical_link_targets.append(inner_index)
                numerical_link_strengths.append(link_strength)
            
pprint(annotation_meta)
len(numerical_link_strengths), 'number of links'

In [ ]:
import matplotlib.pyplot as plt
plt.hist(numerical_link_strengths)

In [ ]:
links_numerical = pd.DataFrame({
    'sources': numerical_link_sources,
    'targets': numerical_link_targets,
    'link_strengths': numerical_link_strengths
})
links_numerical

In [ ]:
points = df[['MivmirScore','case_name']]
points

In [ ]:
from cosmograph import cosmo

In [ ]:
def setup_ui_handler(plot):
    """
    Cosmos UI handler for interacting with points
    """
    def ui_handler(event: dict):
        if event['name'] != 'clicked_point_index':
            return
        point_index = event['new']
        plot.select_point_by_index([point_index])
        plot.focus_point_by_index(point_index)
    
    plot.observe(ui_handler)

In [ ]:
num_plot = cosmo(
    points = points,
    point_label_by = 'case_name',
    point_color_by = 'MivmirScore',
    links = links_numerical,
    link_source_by = 'sources',
    link_target_by = 'targets',
    link_strength_by = 'link_strengths',
    simulation_repulsion=0.1,
    simulation_link_spring=0.01,
    point_size_scale=0.1,
    show_dynamic_labels = True,
    show_labels = True,
    simulation_gravity = 0.1,
    random_seed = 0,
   #link_arrows = True
)
num_plot

In [ ]:
setup_ui_handler(num_plot)

## Text Annotations

In [ ]:
import tensorflow_text as tftext
textual_link_sources = []
textual_link_targets = []
textual_link_strengths = []

text_annotations = ['CLINVAR_GROUND_TRUTH', 'CSQ_SIFT', 'Annotation']
text_annotations = [
    'CSQ_PolyPhen',
    'CSQ_SIFT',
    'CSQ_CLINVAR_CLNREVSTAT',
    'CSQ_CLINVAR_CLNSIG',
    'most_severe_consequence'
]
# > 0.5 means at least 2 matching words, not just 1 [[b'likely', b'benign']]> <tf.RaggedTensor [[b'likely', b'pathogenic']] == 0.5
# Increasing this value will discard less similar matches in favor of identical matches.
threshold = 0.75
threshold = 0.5
#threshold = 0

for annotation in text_annotations:
    if df[annotation].dtype != object:
        continue
    for index, row in df.iterrows():
        for index_inner, row_inner in df.iterrows():
            if index == index_inner:
                continue
            ref = row[annotation]
            # NaN check
            if not ref == ref:
                continue
            hyp = row_inner[annotation]
            if not hyp == hyp:
                continue
            try:
                ref = _preprocess_tensorflow(ref)
                hyp = _preprocess_tensorflow(hyp)
            except Exception as e:
                print(ref, hyp, e)
                continue
            # TODO: check for total length of strings after preprocessing, empty strings should not be considered
            metrics = tftext.metrics.rouge_l(ref, hyp)
            # metrics: F-measure, Precision and Recall
            # Metrics are already in range (0, 1)
            metric = metrics.f_measure.numpy()[0]
            if metric > threshold:
                textual_link_sources.append(index)
                textual_link_targets.append(index_inner)
                textual_link_strengths.append(metric)

links_text = pd.DataFrame({
    'sources': textual_link_sources,
    'targets': textual_link_targets,
    'link_strengths': textual_link_strengths
})
links_text

In [ ]:
txt_plot = cosmo(
    points = points,
    point_label_by = 'case_name',
    point_color_by = 'MivmirScore',
    links = links_text,
    link_source_by = 'sources',
    link_target_by = 'targets',
    link_strength_by = 'link_strengths',
    simulation_repulsion=0.1,
    simulation_link_spring=0.1,
    point_size_scale=0.1,
    show_dynamic_labels = True,
    show_labels = True,
    simulation_gravity = 0.1,
    random_seed = 0,
   #link_arrows = True
)
txt_plot

In [ ]:
setup_ui_handler(txt_plot)

In [ ]:
links_numerical_text = pd.concat((links_numerical, links_text))

# TODO: text features ~1 medans num har lite spridning
links_numerical.hist()
links_text.hist()
#print(links_text.link_strengths.std())
#norm = links_text.link_strengths / links_text.link_strengths.std()
#print(norm)
links_numerical_text.hist()

In [ ]:
num_text_plot = cosmo(
    points = points,
    point_label_by = 'case_name',
    point_color_by = 'MivmirScore',
    links = links_numerical_text,
    link_source_by = 'sources',
    link_target_by = 'targets',
    link_strength_by = 'link_strengths',
    simulation_repulsion=0.05,
    simulation_link_spring=0.001,
    point_size_scale=0.1,
    show_dynamic_labels = True,
    show_labels = True,
    simulation_gravity = 0.0,
    random_seed = 0,
    simulation_decay=1E6
   #link_arrows = True
)
num_text_plot

In [ ]:
setup_ui_handler(num_text_plot)

In [ ]:
facet = sb.relplot(x=df.case_name,
                   y=df.MivmirScore,
                   hue=df.case_name,
                   style=df.case_name,
                   height=FIGSIZE[1],
                   aspect = FIGSIZE[0] / FIGSIZE[1],
                   palette='rocket')
for index, row in df.iterrows():
    facet.ax.text(row.case_name, row.MivmirScore, row.case_name, fontsize=6, rotation=40)
facet.set(xticklabels=[])

# Summary of Network Plots
- It's visible that variants that are less connected (not similar to any other variant) is often misclassified
- The outlier variants are the ones identified as sparsely annotated in previous investigation

In [ ]:
# Visualize what annotations are present in every case
df_annotation_presence = pd.DataFrame(data=np.zeros_like(df.values), columns=sorted(df.columns), index=df.case_name)
df_annotation_presence
for index, row in df.iterrows():
    for annotation in annotations:
        data = row[annotation]
        if isinstance(data, float):
            if data != data:
                continue
        elif isinstance(data, str):
            if row[annotation] == '':
                continue
        else:
            raise ValueError(dtype, row[annotation])
        df_annotation_presence.loc[row.case_name, annotation] = 1  # implies existing annotation

df_annotation_presence = df_annotation_presence.astype(float)
df_annotation_presence['total_annotation'] = df_annotation_presence.sum(axis=1)
df_annotation_presence = df_annotation_presence.sort_values('total_annotation', ascending=True)
df_annotation_presence.drop('total_annotation', inplace=True, axis=1)
fig = plt.figure(figsize=(40, 10))
ax = fig.add_subplot()
sb.heatmap(df_annotation_presence, ax=ax, xticklabels=1, yticklabels=1)
fig.suptitle('Available annotations(0=missing, 1=present)\nCase names in bottom have more annotations.')
fig.tight_layout()

# Summary
- Variants that are less annotated are more likely to have an (incorrect) benign prediction
- Variants that are not annotated with CLINVAR_CLNSIG are more likely to be incorrectly classified as benign
- It's mainly the text features that's driving the sparsity issue, judging from the network plot for textual features.
- Furthermore, missing spliceAI annotations seems to be correlated with low pathogenicity score
- It might be worth syntesizing variants that are sparser in the following annotations, during training:
  - CLINVAR_CLNSIG|CLNREVSTAT
  - GERP++_NR|_RS
  - SpliceAI_*
  - PhastCons100way_vertebrate
  - (PhyloP100way_vertebrate)
  - REVEL_score
  - SIFT
  - SPIDEX